# Medical Insurance Cost Prediction — Regression Models
Using `insurance_cleaned.csv` to train and compare **5 regression algorithms**.

**Pipeline:**
1. Load & inspect cleaned data
2. Feature extraction & engineering
3. Encode categorical features
4. Train/test split & scaling
5. Train 5 regression models
6. Evaluate: MAE, RMSE, R² Score
7. Model comparison visualizations
8. Feature importances
9. Save best model with joblib
10. Actual vs. Predicted values plot

## Step 1 — Import Libraries & Load Data

In [ ]:
import os
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

df = pd.read_csv('../data_cleaning/insurance_cleaned.csv')
print('Dataset shape:', df.shape)
print('Columns:', list(df.columns))
df.head(10)

## Step 2 — Feature Extraction & Engineering

**Features extracted from EDA (data_visualization.ipynb):**

| Feature | Type | Importance |
| --- | --- | --- |
| `smoker` | Categorical | Very High — primary cost driver (~3.8x difference) |
| `obese_smoker` | Engineered | High — obese smokers average \$41,558 vs \$21,363 for non-obese smokers |
| `age` | Numeric | High — correlation r = 0.30 |
| `bmi` | Numeric | Moderate — correlation r = 0.20 |
| `region` | Categorical | Low — southeast highest at \$14,735 avg |
| `sex` | Categorical | Low — slight male premium |
| `children` | Numeric | Very Low — correlation r = 0.07 |

In [ ]:
# Create interaction feature: obese + smoker -> very high risk group
df['obese_smoker'] = ((df['bmi'] >= 30) & (df['smoker'] == 'yes')).astype(int)

print('Obese-smoker count:', df['obese_smoker'].sum())
print('Mean charges by obese_smoker group:')
print(df.groupby('obese_smoker')['charges'].mean().round(2))

## Step 3 — Encode Categorical Features

In [ ]:
df_encoded = pd.get_dummies(df, drop_first=True, dtype=int)

X = df_encoded.drop('charges', axis=1)
y = df_encoded['charges']

print('Final features:', list(X.columns))
print('Feature matrix shape:', X.shape)
X.head()

## Step 4 — Train / Test Split & Feature Scaling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale features for linear models (Linear Regression & Ridge)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print('Training samples:', X_train.shape[0])
print('Test samples:    ', X_test.shape[0])

## Step 5 — Train All 5 Regression Models
Models trained:
1. **Linear Regression**
2. **Decision Tree Regressor**
3. **Random Forest Regressor**
4. **Gradient Boosting Regressor**
5. **Ridge Regression**

In [ ]:
# (name, model_object, use_scaled_data)
models = [
    ('Linear Regression',           LinearRegression(),                     True),
    ('Decision Tree Regressor',     DecisionTreeRegressor(random_state=42), False),
    ('Random Forest Regressor',     RandomForestRegressor(random_state=42), False),
    ('Gradient Boosting Regressor', GradientBoostingRegressor(random_state=42), False),
    ('Ridge Regression',            Ridge(alpha=1.0),                       True),
]

results = []
for name, model, use_scaled in models:
    if use_scaled:
        model.fit(X_train_sc, y_train)
        preds = model.predict(X_test_sc)
    else:
        model.fit(X_train, y_train)
        preds = model.predict(X_test)

    results.append({
        'Model':    name,
        'MAE':      round(mean_absolute_error(y_test, preds), 2),
        'RMSE':     round(root_mean_squared_error(y_test, preds), 2),
        'R2 Score': round(r2_score(y_test, preds), 4),
    })
    print(f'{name:35s} -> MAE={results[-1]["MAE"]:>10,.2f}  RMSE={results[-1]["RMSE"]:>10,.2f}  R2={results[-1]["R2 Score"]:.4f}')

print('\nAll models trained successfully.')

## Step 6 — Evaluation Metrics Table

In [ ]:
results_df = pd.DataFrame(results).sort_values('R2 Score', ascending=False).reset_index(drop=True)
results_df.index += 1   # rank starts at 1
results_df

## Step 7 — Model Comparison Visualizations
### Chart 1: R² Score (Higher is Better)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336']
bars = ax.bar(results_df['Model'], results_df['R2 Score'], color=colors, edgecolor='black', width=0.5)
ax.set_title('Model Comparison — R² Score (Higher is Better)', fontsize=14, fontweight='bold', pad=12)
ax.set_ylabel('R² Score', fontsize=12)
ax.set_ylim(0, 1.12)
ax.tick_params(axis='x', rotation=15)
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.axhline(y=0.9, color='red', linestyle='--', linewidth=1.2, alpha=0.7, label='R²=0.90 threshold')
ax.legend(fontsize=10)
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, h + 0.018,
            f'{h:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()

### Chart 2: MAE — Mean Absolute Error (Lower is Better)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colors_mae = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336']
bars = ax.bar(results_df['Model'], results_df['MAE'], color=colors_mae, edgecolor='black', width=0.5)
ax.set_title('Model Comparison — MAE (Lower is Better)', fontsize=14, fontweight='bold', pad=12)
ax.set_ylabel('Mean Absolute Error ($)', fontsize=12)
ax.set_ylim(0, results_df['MAE'].max() * 1.25)
ax.tick_params(axis='x', rotation=15)
ax.grid(axis='y', alpha=0.3, linestyle='--')
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, h + 40,
            f'${h:,.0f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()

### Chart 3: RMSE — Root Mean Squared Error (Lower is Better)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colors_rmse = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336']
bars = ax.bar(results_df['Model'], results_df['RMSE'], color=colors_rmse, edgecolor='black', width=0.5)
ax.set_title('Model Comparison — RMSE (Lower is Better)', fontsize=14, fontweight='bold', pad=12)
ax.set_ylabel('Root Mean Squared Error ($)', fontsize=12)
ax.set_ylim(0, results_df['RMSE'].max() * 1.25)
ax.tick_params(axis='x', rotation=15)
ax.grid(axis='y', alpha=0.3, linestyle='--')
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, h + 80,
            f'${h:,.0f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.show()

### Chart 4: Side-by-Side — All 3 Metrics Combined

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Model Comparison — All Evaluation Metrics', fontsize=14, fontweight='bold')

metric_info = [
    ('R2 Score', 'R² Score (Higher is Better)', 'steelblue',  None),
    ('MAE',      'MAE in $ (Lower is Better)',  'seagreen',   '$'),
    ('RMSE',     'RMSE in $ (Lower is Better)', 'darkorange', '$'),
]

for ax, (col, title, color, prefix) in zip(axes, metric_info):
    b = ax.bar(results_df['Model'], results_df[col], color=color, edgecolor='black')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_ylim(0, results_df[col].max() * 1.3)
    ax.tick_params(axis='x', rotation=25, labelsize=8)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    for bar in b:
        h = bar.get_height()
        label = f'{prefix}{h:,.0f}' if prefix else f'{h:.4f}'
        ax.text(bar.get_x() + bar.get_width() / 2, h + results_df[col].max() * 0.02,
                label, ha='center', va='bottom', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.show()

## Step 8 — Feature Importances (Gradient Boosting)

In [ ]:
gb_model = models[3][1]  # Gradient Boosting Regressor
importances = pd.Series(gb_model.feature_importances_, index=X.columns).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(importances.index, importances.values, color='mediumpurple', edgecolor='black')
ax.set_title('Feature Importances — Gradient Boosting Regressor', fontsize=13, fontweight='bold')
ax.set_ylabel('Importance Score', fontsize=11)
ax.tick_params(axis='x', rotation=25)
ax.grid(axis='y', alpha=0.3, linestyle='--')
for bar in bars:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, h + 0.005,
            f'{h:.4f}', ha='center', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nFeature Importances (Gradient Boosting):')
print(importances.round(4).to_string())

## Step 9 — Save Best Model with Joblib

Saves the best model (**Linear Regression**, R² = 0.9057) and `StandardScaler` to `saved_models/`, then reloads and verifies the R² score.

In [ ]:
best_model = models[0][1]
best_model_name = models[0][0]

os.makedirs('saved_models', exist_ok=True)
joblib.dump(best_model, 'saved_models/linear_regression_best_model.pkl')
joblib.dump(scaler,     'saved_models/scaler.pkl')

print('Best model saved : saved_models/linear_regression_best_model.pkl')
print('Scaler saved     : saved_models/scaler.pkl')
print('Model name       :', best_model_name)

loaded_model  = joblib.load('saved_models/linear_regression_best_model.pkl')
loaded_scaler = joblib.load('saved_models/scaler.pkl')

preds_check = loaded_model.predict(loaded_scaler.transform(X_test))
print(f'Verification R2  : {r2_score(y_test, preds_check):.4f}')

## Step 10 — Actual vs. Predicted Values Plot (Best Model: Linear Regression)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(y_test, preds_check, alpha=0.6, color='steelblue', edgecolor='black', linewidth=0.4, label='Test Predictions')

min_val = min(y_test.min(), preds_check.min())
max_val = max(y_test.max(), preds_check.max())
ax.plot([min_val, max_val], [min_val, max_val], color='red', linestyle='--', linewidth=1.8, label='Perfect Prediction (y = x)')

ax.set_title('Actual vs. Predicted Charges — Linear Regression (R2 = 0.9057)', fontsize=13, fontweight='bold')
ax.set_xlabel('Actual Charges ($)', fontsize=11)
ax.set_ylabel('Predicted Charges ($)', fontsize=11)
ax.legend()
ax.grid(alpha=0.3, linestyle='--')
plt.tight_layout()
plt.show()

## Conclusion

**Model Ranking (by R² Score):**

| Rank | Model | MAE | RMSE | R² Score |
| --- | --- | ---: | ---: | ---: |
| 1 | **Linear Regression** | \$2,391.56 | \$4,163.41 | **0.9057** |
| 2 | **Ridge Regression** | \$2,392.82 | \$4,164.45 | **0.9056** |
| 3 | **Gradient Boosting Regressor** | \$2,463.40 | \$4,217.58 | **0.9032** |
| 4 | **Random Forest Regressor** | \$2,612.97 | \$4,681.71 | **0.8807** |
| 5 | **Decision Tree Regressor** | \$2,843.50 | \$6,098.73 | **0.7976** |

**Key Insights:**
- **Best Model:** Linear Regression (R² = 0.9057) — explains 90.6% of cost variance.
- **Why linear models win:** The engineered `obese_smoker` feature linearises the dominant non-linear interaction between BMI and smoking status.
- **Top 3 features** (Gradient Boosting): `obese_smoker` (68.78%), `smoker_yes` (14.15%), `age` (12.37%) — together account for **95.3%** of predictive power.
- **Decision Tree** overfits without pruning, giving the worst generalisation (R² = 0.7976).